In [ ]:
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
#läser datat 
data = pd.read_csv("data_grupp_29.csv")
print("Reder och kolumner: ", data.shape)
data.head()

In [ ]:
data.info()

In [ ]:
#Ser om det finns dubbletter och tomma värden
print("Saknade värden per kolumn:")
print (data.isna().sum())
print("\n Antal dubblettrader: ",data.duplicated().sum())

In [ ]:
print(data["target"].value_counts())
data["target"].value_counts().plot(kind="bar",  title ="Antal prover per klass")
plt.ylabel("Antal"); plt.xticks(rotation=0); plt.show()

In [ ]:
#Delar upp värden i två lådor 
x = data.drop(columns="target")
y = data["target"]

In [ ]:
x.describe().T.round(2)

In [ ]:
#skapar en histogram med med storlek 
x.hist(figsize= (14, 8), bins= 20)
#Sapar rubriker och hindrar så att bilden och texten inte krockar 
plt.suptitle("Fördekning per variable")
plt.tight_layout(); plt.show

In [ ]:
#Skapar en X  med rätt 
plt.figure(figsize=(12,5))
x.boxplot()
plt.title("Boxplot per variabel(oskalad data)")
plt.xticks(rotation=30);plt.show()

In [ ]:
q1 = x.quantile(0.25)
q3 = x.quantile(0.75)
iqr = q3 - q1
outliers =((x < q1-1.5 * iqr)| (x > q3 + 1.5 * iqr)).sum()
outliers.to_frame("antal outliers")

In [ ]:
#Jämföer tre lag sida vid sida 
fig, axes = plt.subplots(2,3, figsize = (15,8))
ordning = ["God_status", "Mattlig_paverkan", "Overgodd"]
for ax, kol in zip(axes.ravel(), x.columns): 
    sns.boxplot(data=data, x="target", y=kol, order=ordning, ax=ax)
    ax.set_title(kol); ax.set_xlabel("")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
#skalar värdet 
scaler = StandardScaler() # skapar verktyg 
xs = scaler.fit_transform(x) # lär dig medel/std + räknar om

#kontrolerar
pd.DataFrame(xs, columns=x.columns).describe().T[["mean", "std"]].round(2)

In [ ]:
jamfor = pd.DataFrame(xs, columns=x.columns )
print("Före skalning:"); print (x.iloc [[1,2]] [["ph_varde", "konduktivitet_us_cm"]])
print("Efter skalning:"); print (jamfor.iloc [[1,2]] [["ph_varde", "konduktivitet_us_cm"]].round(2))

In [ ]:
## A2 Unsupervised screening (utan target, standardinställninger)

In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
km = KMeans(random_state=42).fit(xs)
km_labels = km.labels_
print("k-means kluster:", len(np.unique(km_labels)))
print("Silhouette:", round(silhouette_score(xs, km_labels), 3))

In [ ]:
#A2---DBSCAN med standardinställningar--
# Visar antal kluster, andel brus(-1) och silhouette beräknad utan brus- punkter.
db = DBSCAN().fit(xs) 
db_labels = db.labels_
mask = db_labels !=-1
n_kluster = len(set(db_labels[mask]))
print("DBSCAN kluster:" , n_kluster)
print(f"Brus: {(~mask).mean()*100:.1f} %")
if n_kluster >= 2:
    print("silhouette (utan brus):", round(silhouette_score(xs[mask], db_labels[mask]),3))
else:
    print("Silhouette kan inte räknas (färre än 2 kluster)")
    

In [ ]:
pca = PCA(n_components=2, random_state=42)
xp= pca.fit_transform(xs)
print("Förklarad varians:" , pca.explained_variance_ratio_.round(3))
print("Totalt PC1 + PC2:", round(pca.explained_variance_ratio_.sum(), 3))


In [ ]:
y_kod = y.astype("category" ).cat.codes
fig, ax =plt.subplots(1,3, figsize=(17, 5))
ax[0].scatter(xp[:,0], xp[: ,1], c=km_labels,cmap="tab10", s=15); ax [0].set_title("k-means(k=8)")
ax[1].scatter(xp[mask,0], xp[mask,1], c=db_labels[mask], cmap="tab10", s=15)
ax[1].scatter(xp[~mask,0], xp[~mask,1], c="lightgrey",s=15, label="brus"); ax[1].legend()
ax[1].set_title("DBSCAN (standard)")
ax[2].scatter(xp[:,0], xp[:,1], c=y_kod, cmap="viridis", s=15); ax[2].set_title("Riktiga klasser")
for a in ax: a.set_xlabel("PC1"); a.set_ylabel("PC2")
plt.tight_layout(); plt.show()


In [ ]:
## SKriv en observation 

In [ ]:
###A3 Supervised screening (5-fold stratfied CV, standardinställningar)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

In [ ]:

modeller = {
    "KNN": KNeighborsClassifier(),
    "SVM": SVC (random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}
cv5 = StratifiedKFold (n_splits=5, shuffle=True, random_state=42)

In [ ]:
rader =[]
for namn, modell in modeller.items():
    pipe = make_pipeline(StandardScaler(), modell )
    r = cross_validate(pipe, x, y, cv=cv5, scoring=["accuracy", "f1_macro"])
    rader.append({
        "Modell" : namn,
        "Accuracy medel": r["test_accuracy"].mean(),
        "Accuracy std": r["test_accuracy"].std(),
        "F1 macro medel": r["test_f1_macro"].mean(),
        "F1 macro std": r["test_f1_macro"].std(),
        "Tid(s)": r["fit_time"].mean(),

    })
    screening = pd.DataFrame(rader).set_index("Modell").round(3)
    screening.sort_values("F1 macro medel", ascending=False)

In [ ]:

## Ritar ett stapeldiagrm över screeningresultaten, med Accuracy och F1 macro för alla fem modeller.
ax = screening[["Accuracy medel", "F1 macro medel"]].plot(
    kind="bar", yerr=screening[["Accuracy std", "F1 macro std"]].values.T,
    figsize=(9,4), capsize=3, rot=0)
ax.set_ylim(0, 1.05)
ax.set_title("Screening: 5-fold Stratified CV (medel ± std)")
plt.tight_layout(); plt.show()


